# 04 — Machine learning sur variables décalées

Autre approche : transformer la série en table (une ligne = une semaine, colonnes = valeurs
passées et repères de calendrier) et confier ça à un régresseur classique. Le détail est
dans `src/features.py` et `src/models.py` (classe `MLForecaster`, prévision récursive).

In [1]:
import sys, warnings, os
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import preprocessing, features, models, plotting
s = preprocessing.serie_hebdo(3, 'PAY').dropna()

In [2]:
X, y = features.supervise(s, horizon=1)
print("table supervisée :", X.shape)
X.head(3)

table supervisée : (1828, 19)


,niveau,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_52,moy_3,tendance_3,moy_8,tendance_8,diff_1,accel,sem_sin,sem_cos,saison_epi
1991-08-05,1.0,4.0,4.0,7.0,6.0,6.0,11.0,14.0,14.0,2.0,3.000000,-2.000000,6.625,-1.625,-3.0,-3.0,-0.663123,-0.748511,0
1991-08-12,3.0,1.0,4.0,4.0,7.0,6.0,6.0,11.0,14.0,0.0,2.666667,-0.333333,5.250,-1.375,2.0,5.0,-0.748511,-0.663123,0
1991-08-19,4.0,3.0,1.0,4.0,4.0,7.0,6.0,6.0,11.0,0.0,2.666667,0.000000,4.375,-0.875,1.0,-1.0,-0.822984,-0.568065,0


Les features : décalages de 1 à 8 semaines et 52 semaines, moyennes et pentes glissantes
(3 et 8 semaines), variation et accélération récentes, et la semaine ISO encodée en
sin/cos. Rien qui utilise le futur.

In [3]:
# Découpage temporel : on entraîne sur l'avant-2019, on évalue en glissant sur l'après.
coupe = "2019-07-01"
Xtr, ytr = X.loc[:coupe], y.loc[:coupe]

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
gb = HistGradientBoostingRegressor(learning_rate=0.05, max_depth=3, max_iter=400,
                                   l2_regularization=1.0, random_state=0).fit(Xtr, ytr)
imp = permutation_importance(gb, X.loc[coupe:], y.loc[coupe:], n_repeats=10, random_state=0)
pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False).head(10).round(3)

niveau        0.510
tendance_8    0.191
sem_cos       0.011
lag_8         0.009
sem_sin       0.009
diff_1        0.006
lag_7         0.006
tendance_3    0.004
lag_6         0.004
lag_1         0.004
dtype: float64

`niveau` (la valeur de la semaine courante) domine très largement, suivi de `tendance_8` — le fait d'être en phase de montée ou de descente. À une semaine, l'essentiel de l'information tient dans ces deux variables ; les décalages plus anciens et le calendrier n'ajoutent presque rien.

In [4]:
# Évaluation courte glissante (t+1 et t+4), comme au notebook 03, pour comparer sur la
# même base. Réutilise src.backtest.
from src import backtest
tables = {}
for nom, ctor in [("Naïf saisonnier", models.SeasonalNaive),
                  ("SARIMA", models.Sarima),
                  ("Ridge (lags)", models.ridge),
                  ("Gradient boosting (lags)", models.gradient_boosting)]:
    r = backtest.rolling_origin(ctor, s, debut_test="2019-07-01", horizons=(1, 4),
                                pas=2, refit_every=26)
    tables[nom] = backtest.scores(r).drop(columns="RMSE")
comparaison = pd.concat({k: v["MAE"] for k, v in tables.items()}, axis=1)
comparaison.index = [f"+{h} sem." for h in comparaison.index]
comparaison.round(2)

,Naïf saisonnier,SARIMA,Ridge (lags),Gradient boosting (lags)
+1 sem.,44.26,13.30,15.40,14.41
+4 sem.,45.20,42.52,56.02,50.29


Résultat : à **1 semaine, les modèles ML sur décalages font jeu égal avec SARIMA** (MAE ~14-15 contre ~13). À 4 semaines ils décrochent nettement (~50-56 contre ~43) : la prévision récursive accumule l'erreur. Le gradient boosting ne bat pas la régression Ridge — pas de non-linéarité à exploiter une fois les bons décalages fournis.

L'intérêt de cette approche est surtout qu'elle accueille facilement des régresseurs externes (météo, vacances scolaires, recherches Google) sans changer de cadre. Le backtest complet du notebook 06 confirme : au coude à coude avec SARIMA à court terme, en retrait ensuite.